# 00 — Viết qrels (bảng chân lý cho retrieval)

**qrels = bảng chân lý**: với mỗi câu hỏi, bài nào trả lời được, bài nào chỉ liên quan, bài nào không.

Không có qrels thì mọi metric (MRR, nDCG@k) chỉ là con số vô nghĩa — không biết đúng sai.
Đây là file bạn sẽ quay lại nhiều lần: viết 12 query trước, sau đó mở rộng lên 40-50.

## Thang điểm (áp dụng NHẤT QUÁN cho mọi query)

| Điểm | Nghĩa | Cách tự hỏi |
|---|---|---|
| **2** | Bài **trả lời trực tiếp** câu hỏi | "Đọc xong bài này, người hỏi có biết đáp án không?" |
| **1** | **Liên quan một phần** — cùng chủ đề, có nhắc tới, nhưng không trả lời trực tiếp | "Bài này nói về chủ đề khác nhưng có nhắc tới khái niệm trong câu hỏi" |
| **0** | Không liên quan (chỉ chấm khi bạn đã thực sự xem bài đó) | "Bài này chẳng giúp gì cho câu hỏi" |

## Quy trình 3 bước cho MỖI câu hỏi

1. `search("từ khoá")` → xem bài ứng viên
2. `show("<doc_id>")` → đọc toàn văn bài nghi ngờ
3. `judge(...)` → chấm điểm

Cuối cùng chạy `save()` để ghi ra `eval/qrels.csv`.

In [1]:
# ── Khởi tạo: nạp corpus + công cụ ─────────────────────────────
import json, unicodedata
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORPUS = ROOT / "data" / "processed" / "curated.jsonl"
QRELS = ROOT / "eval" / "qrels.csv"

docs = pd.DataFrame([json.loads(l) for l in CORPUS.read_text(encoding="utf-8").splitlines() if l.strip()])
docs["doc_id"] = docs["doc_id"].astype(str)
print(f"Corpus: {len(docs)} bài · {docs['chars'].sum()/1e6:.2f} MB text")
docs[["doc_id", "title", "chars"]].head(3)

Corpus: 179 bài · 2.47 MB text


,doc_id,title,chars
0,20015911,AI Overviews,6752
1,19299575,AI-đầy đủ,1227
2,19991504,AlexNet,6611


In [2]:
# ── Công cụ (chạy 1 lần, không cần sửa) ────────────────────────
def _strip_accents(t: str) -> str:
    t = t.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")

def search(term: str, k: int = 8, snippet: int = 150):
    """Tìm bài ứng viên — gõ có dấu hay không dấu đều được."""
    pat = _strip_accents(term.lower())
    rows = []
    for _, d in docs.iterrows():
        text = _strip_accents(d["text"].lower())
        title = _strip_accents(d["title"].lower())
        n = text.count(pat)
        if n or pat in title:
            i = text.find(pat)
            start = max(0, i - snippet // 2)
            snip = d["text"][start:start + snippet].replace("\n", " ")
            rows.append({"doc_id": d["doc_id"], "title": d["title"], "chars": d["chars"],
                         "hits": n, "title_hit": pat in title, "snippet": "..." + snip + "..."})
    out = pd.DataFrame(rows).sort_values(["title_hit", "hits"], ascending=False).head(k)
    return out[["doc_id", "title", "chars", "hits", "snippet"]] if len(out) else "Không tìm thấy bài nào."

def show(doc_id, limit: int = 2000):
    """In toàn văn (hoặc `limit` ký tự đầu) một bài để chấm điểm."""
    row = docs[docs["doc_id"] == str(doc_id)]
    if row.empty:
        print(f"Không có doc_id {doc_id}"); return
    r = row.iloc[0]
    print(f"# {r['title']}\n{r['url']}\n({r['chars']:,} ký tự)\n")
    print(r["text"][:limit] + ("\n\n[... còn nữa — tăng limit để đọc thêm]" if r["chars"] > limit else ""))

# ── Sổ chấm điểm (nạp lại nếu file đã có) ──────────────────────
if QRELS.exists():
    jud = pd.read_csv(QRELS, dtype={"doc_id": str}).dropna(subset=["relevance"]).to_dict("records")
    print(f"Đã nạp {len(jud)} dòng có sẵn từ {QRELS.relative_to(ROOT)}")
else:
    jud = []
    print("Chưa có file qrels — sẽ tạo mới khi save()")

def judge(query_id: str, query: str, grades: dict, query_class: str):
    """Chấm điểm cho MỘT câu hỏi. grades = {"doc_id": điểm, ...}

    ví dụ: judge("q001", "Học có giám sát là gì?", {"24329": 2, "19895408": 1}, "literal")
    """
    assert query_class in {"literal", "paraphrase", "multi-doc"}, "query_class phải là literal / paraphrase / multi-doc"
    for did, rel in grades.items():
        assert str(did) in set(docs["doc_id"]), f"doc_id {did} không tồn tại trong corpus!"
        assert rel in (0, 1, 2), "relevance chỉ nhận 0 / 1 / 2"
        jud.append({"query_id": query_id, "query": query, "doc_id": str(did),
                    "relevance": rel, "query_class": query_class})
    print(f"✓ {query_id}: chấm {len(grades)} bài  ({query_class})")

def progress():
    """Tiến độ đợt 1: cần 12 query (4 literal + 4 paraphrase + 4 multi-doc)."""
    if not jud:
        print("Chưa chấm query nào."); return
    df = pd.DataFrame(jud)
    per_q = df.groupby(["query_id", "query_class"]).agg(số_bài=("doc_id", "count")).reset_index()
    print(per_q.to_string(index=False))
    print("\nTheo lớp:", df.groupby("query_class")["query_id"].nunique().to_dict())
    print(f"Tổng: {df['query_id'].nunique()} query · {len(df)} dòng")

def save():
    """Ghi ra eval/qrels.csv (giữ định dạng đọc được bằng pandas)."""
    if not jud:
        print("Chưa chấm query nào — không ghi đè file qrels.")
        return
    df = pd.DataFrame(jud)[["query_id", "query", "doc_id", "relevance", "query_class"]]
    QRELS.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(QRELS, index=False)
    print(f"Đã lưu {len(df)} dòng → {QRELS.relative_to(ROOT)}")
print("Công cụ sẵn sàng: search() · show() · judge() · progress() · save()")

Đã nạp 0 dòng có sẵn từ eval\qrels.csv
Công cụ sẵn sàng: search() · show() · judge() · progress() · save()


## Bước 1 — Tìm bài ứng viên

Gõ từ khoá **của chủ đề** (không phải cả câu hỏi). Ví dụ câu hỏi là *"Học có giám sát là gì?"*
thì từ khoá để tra là `học có giám sát`.

In [3]:
search("học có giám sát")

,doc_id,title,chars,hits,snippet
2,24329,Học có giám sát,7651,6,...Học có giám sát là một kĩ thuật của ngành h...
18,20014464,Thuật ngữ trí tuệ nhân tạo,42756,7,"...tác vụ hoặc miền khác có liên quan, giúp gi..."
8,19895408,Học tự giám sát,8010,4,"...khởi tạo các tham số của mô hình. Sau đó, t..."
10,19893713,Học đặc trưng,24688,4,"...c đặc trưng có thể là có giám sát, không gi..."
5,3339820,Học sâu,57004,3,...nghĩa này có điểm chung là (1) nhiều lớp cá...
14,16760383,Mạng thần kinh,15653,3,"...u học chính, mỗi kiểu mẫu tương ứng với một..."
3,24345,Học không có giám sát,1866,2,... máy nhằm tìm ra một mô hình mà phù hợp với...
4,24278,Học máy,19947,2,...ong quá trình huấn luyện. Theo truyền thống...


## Bước 2 — Đọc bài nghi ngờ

Bài nào tiêu đề đúng chủ đề thì gần như chắc chắn điểm 2. Bài nào chỉ *nhắc tới* thì phải đọc mới biết
→ dùng `show()`.

In [4]:
show("19895408", limit=1200)

# Học tự giám sát
https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_t%E1%BB%B1_gi%C3%A1m_s%C3%A1t
(8,010 ký tự)

Học tự giám sát (self-supervised learning hay SSL) là một kỹ thuật trong học máy mà trong đó, một mô hình được huấn luyện trên một tác vụ bằng cách sử dụng chính dữ liệu để tạo ra các tín hiệu giám sát, thay vì dựa vào các nhãn bên ngoài do con người cung cấp. Trong bối cảnh của mạng thần kinh nhân tạo, học tự giám sát hướng tới việc khai thác các cấu trúc hoặc mối quan hệ nội tại trong dữ liệu đầu vào để tạo ra các tín hiệu huấn luyện có ý nghĩa. Các nhiệm vụ trong SSL được thiết kế sao cho việc giải quyết yêu cầu nắm bắt được các đặc điểm hoặc mối quan hệ quan trọng trong dữ liệu. Dữ liệu đầu vào thường được tăng cường hoặc biến đổi để tạo ra các cặp mẫu liên quan. Một mẫu sẽ đóng vai trò làm đầu vào, và mẫu còn lại được dùng để hình thành tín hiệu giám sát. Việc tăng cường này có thể bao gồm thêm nhiễu, cắt, xoay, hoặc các biến đổi khác. Học tự giám sát mô phỏng cách con người họ

## Bước 3 — Chấm điểm

Mỗi câu hỏi = 1 lệnh `judge()`. Truyền dict `{doc_id: điểm}`.

Ví dụ dưới đây là **câu hỏi mẫu của bro**, bạn sửa lại theo phán đoán của CHÍNH BẠN rồi bỏ dấu `#` để chạy.

In [5]:
# judge("q001", "Học có giám sát là gì?", {
#     "24329":    2,   # bài đúng chủ đề, định nghĩa trực tiếp
#     "19895408": 1,   # "Học tự giám sát" — chủ đề khác, chỉ nhắc để so sánh
#     "20014464": 1,   # "Thuật ngữ AI" — có 1 dòng định nghĩa, không giải thích sâu
# }, "literal")

progress()

Chưa chấm query nào.


## Checklist 12 query đợt 1

- [ ] 4 × **literal** — dùng đúng từ khoá trong bài (ví dụ: *"Mạng nơ-ron tích chập là gì?"*)
- [ ] 4 × **paraphrase** — KHÔNG lặp từ khoá của bài (ví dụ: *"Máy tính học từ ví dụ có nhãn bằng cách nào?"* thay vì *"học có giám sát"*)
- [ ] 4 × **multi-doc** — cần ≥2 bài, gold phải có nhiều doc (ví dụ: *"So sánh học có giám sát và học tự giám sát"*)

**Mẹo chọn chủ đề:** mở `python scripts/explore.py --titles` ở terminal để xem 179 tiêu đề,
chọn chủ đề bạn đã học (chấm relevance tự tin hơn nhiều).

In [6]:
# ── Lưu lại ────────────────────────────────────────────────────
progress()
save()

Chưa chấm query nào.
Chưa chấm query nào — không ghi đè file qrels.
